In [2]:
%load_ext autoreload

```python
import ollama
```


```python
%autoreload 2
from mabot.agents.task.planner import ScreenshotPlannerAgent
from mabot.browser.browser import AgentBrowser
```


```python
selector = WebElementSelectorByCursorCss(
    style=["pointer", "text"], is_displayed=True
)
browser = AgentBrowser(
    "https://www.google.com/",
    selector=selector,
    font_size=50,
    label_margin=5,
)
```


```python
screen = browser.markup.screenshot_image()
screen
```


```python
markup = browser.markup.markup_tiles(margin=2)
markup.view
```


```python
screen.save("test.png")
```


```python
prompt = """
You are a browser operator.
Given view with your current browser state plan next steps on how to solve the task.
Each step performs actions on the element.
First step must start with current view.
Each step should be presented strictly in the format:

**Step Format**
*number*: *action*

**Actions**
- Click[element]
- TypeText[element, text]
- GoBack
- ScrollUp
- ScrollDown

**Task**
Find how to import python modules on stackoverflow.

Respond STRICTLY in this format:
**Your response**
*Thought*: {{Briefly summarize information that will help to solve the task}}
*Steps*: {{number: }}
"""

response = ollama.chat(
    model="llama3.2-vision",
    messages=[
        {
            "role": "user",
            "content": prompt,
            "images": ["test.png"],
        }
    ],
)

# print(response)
```


```python
prompt = """
You are a browser operator.
You are given the image of the current website view with grid.
Each grid cell has its identification in the top-left corner.
Website elements on the image are positioned using the grid.

Give all web elements on this image which intersect with grid cell ID=32
"""

response = ollama.chat(
    model="llama3.2-vision",
    messages=[
        {
            "role": "user",
            "content": prompt,
            "images": ["test.png"],
        }
    ],
)

# print(response)
```


```python
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig

# load the processor
processor = AutoProcessor.from_pretrained(
    "allenai/Molmo-7B-O-0924",
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)

# load the model
model = AutoModelForCausalLM.from_pretrained(
    "allenai/Molmo-7B-O-0924",
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)

# process the image and text
inputs = processor.process(images=[markup.view], text=prompt)

# move inputs to the correct device and make a batch of size 1
inputs = {k: v.to(model.device).unsqueeze(0) for k, v in inputs.items()}

# generate output; maximum 200 new tokens; stop generation when <|endoftext|> is generated
output = model.generate_from_batch(
    inputs,
    GenerationConfig(max_new_tokens=200, stop_strings="<|endoftext|>"),
    tokenizer=processor.tokenizer,
)

# only get generated tokens; decode them to text
generated_tokens = output[0, inputs["input_ids"].size(1) :]
generated_text = processor.tokenizer.decode(
    generated_tokens, skip_special_tokens=True
)

# print the generated text
```


```python
agent = ScreenshotPlannerAgent().get_ollama_agent("llama3.2-vision")
```


```python
from langchain.globals import set_verbose

set_verbose(True)

agent.invoke(
    {
        "image": [browser.markup.screenshot_image_base64()],
        "task": "Find how to import python modules on stackoverflow",
    }
)
```


```python
prompt = """
You are a web browser user. Analyse current view and plan next steps on how to solve the task.
Respond with the numbered list of the steps.

**Task**
Find how to import python modules on stackoverflow.

*Next steps*
"""

response = ollama.chat(
    model="llama3.2-vision",
    messages=[
        {
            "role": "user",
            "content": prompt,
            "images": ["test.png"],
        }
    ],
)
```
